# Bluestock Mutual Fund Analytics Capstone - Day 2: Database Load & Analytical Queries

## Day 2 Objectives:
1. **Load cleaned data** into the SQLite database (`data/db/bluestock_mf.db`).
2. **Populate all Star Schema tables** defined in the `schema.sql` file.
3. **Write and test 10 analytical SQL queries** to extract insights from the data.

In [1]:
import os
sqlite_db_path = r'x:\CODING\PROJECTS\InternshipWork\Bluestock\Capstone\data\db\bluestock_mf.db'
if os.path.exists(sqlite_db_path):
    try:
        os.remove(sqlite_db_path)
        print('Deleted old database file for a clean load.')
    except Exception as e:
        print(f'Warning: could not delete database: {e}')

Deleted old database file for a clean load.


In [2]:
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine

# Paths
base_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
processed_dir = os.path.join(base_dir, "data", "processed")
db_dir = os.path.join(base_dir, "data", "db")
db_path = os.path.join(db_dir, "bluestock_mf.db")
schema_path = os.path.join(base_dir, "sql", "schema.sql")

print(f"Database Path: {db_path}\nSchema Path: {schema_path}")

Database Path: X:\CODING\PROJECTS\InternshipWork\Bluestock\Capstone\data\db\bluestock_mf.db
Schema Path: X:\CODING\PROJECTS\InternshipWork\Bluestock\Capstone\sql\schema.sql


### 1. Initialize Database Schema

In [3]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("Initializing database tables...")
with open(schema_path, 'r') as f:
    schema_sql = f.read()
    
conn.executescript(schema_sql)
conn.commit()
print("Schema initialized successfully.")

Initializing database tables...
Schema initialized successfully.


### 2. Populate Dimensions and Facts

In [4]:
engine = create_engine(f"sqlite:///{db_path}")

# Load data
df_fund = pd.read_csv(os.path.join(processed_dir, "01_fund_master_clean.csv"))
df_nav = pd.read_csv(os.path.join(processed_dir, "02_nav_history_clean.csv"))
df_aum = pd.read_csv(os.path.join(processed_dir, "03_aum_by_fund_house_clean.csv"))
df_sip = pd.read_csv(os.path.join(processed_dir, "04_monthly_sip_inflows_clean.csv"))
df_perf = pd.read_csv(os.path.join(processed_dir, "07_scheme_performance_clean.csv"))
df_tx = pd.read_csv(os.path.join(processed_dir, "08_investor_transactions_clean.csv"))
df_port = pd.read_csv(os.path.join(processed_dir, "09_portfolio_holdings_clean.csv"))

print("Generating and inserting Date Dimension...")
# Collect unique dates to populate dim_date
all_dates = pd.concat([
    df_nav['date'],
    df_tx['transaction_date'],
    df_aum['date']
]).dropna().unique()

df_date = pd.DataFrame({'date': all_dates})
df_date['date'] = pd.to_datetime(df_date['date'])
df_date = df_date.sort_values('date')
df_date['year'] = df_date['date'].dt.year
df_date['month'] = df_date['date'].dt.month
df_date['quarter'] = df_date['date'].dt.quarter
df_date['is_weekday'] = df_date['date'].dt.weekday.map(lambda x: 1 if x < 5 else 0)
df_date['date'] = df_date['date'].dt.strftime('%Y-%m-%d')

df_date.to_sql('dim_date', con=engine, if_exists='append', index=False)
print(f"Inserted {len(df_date)} rows into dim_date")

Generating and inserting Date Dimension...
Inserted 1608 rows into dim_date


In [5]:
print("Populating dim_fund...")
df_fund.to_sql('dim_fund', con=engine, if_exists='append', index=False)

print("Populating fact_nav...")
df_nav.to_sql('fact_nav', con=engine, if_exists='append', index=False)

print("Populating fact_transactions...")
# Map column names to schema
df_tx_db = df_tx.rename(columns={
    'transaction_date': 'date',
    'amount_inr': 'amount',
    'transaction_type': 'type'
})
df_tx_db.to_sql('fact_transactions', con=engine, if_exists='append', index=False)

print("Populating fact_performance...")
# Map columns
df_perf_db = df_perf.rename(columns={
    'return_1yr_pct': 'return_1yr',
    'return_3yr_pct': 'return_3yr',
    'return_5yr_pct': 'return_5yr',
    'sharpe_ratio': 'sharpe',
    'sortino_ratio': 'sortino',
    'max_drawdown_pct': 'max_dd'
})
# Add as_of_date
df_perf_db['as_of_date'] = '2026-05-31'

# Select only columns matching schema
perf_columns = [
    'amfi_code', 'as_of_date', 'return_1yr', 'return_3yr', 'return_5yr', 
    'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe', 'sortino', 
    'std_dev_ann_pct', 'max_dd', 'morningstar_rating'
]
df_perf_db = df_perf_db[perf_columns]
df_perf_db.to_sql('fact_performance', con=engine, if_exists='append', index=False)

print("Populating fact_portfolio...")
# Map columns and filter to schema
df_port_db = df_port.rename(columns={'portfolio_date': 'date'})
port_columns = ['amfi_code', 'stock_symbol', 'weight_pct', 'sector', 'date']
df_port_db = df_port_db[port_columns]
df_port_db.to_sql('fact_portfolio', con=engine, if_exists='append', index=False)

print("Populating fact_aum...")
# Select columns matching schema
df_aum_db = df_aum[['fund_house', 'date', 'aum_crore', 'num_schemes']]
df_aum_db.to_sql('fact_aum', con=engine, if_exists='append', index=False)

print("Populating fact_sip_industry...")
df_sip.to_sql('fact_sip_industry', con=engine, if_exists='append', index=False)

print("All tables populated successfully!")

Populating dim_fund...
Populating fact_nav...


Populating fact_transactions...


Populating fact_performance...
Populating fact_portfolio...
Populating fact_aum...
Populating fact_sip_industry...
All tables populated successfully!


### 3. Run and Test 10 Analytical SQL Queries

In [6]:
def run_query(title, query):
    print("="*80)
    print(f"QUERY: {title}")
    print("="*80)
    df = pd.read_sql_query(query, engine)
    display(df)
    print("\n")

In [7]:
# Query 1: Top 5 Fund Houses by Total AUM in the Latest Period
run_query("Top 5 Fund Houses by Total AUM", """
SELECT fund_house, date, aum_crore 
FROM fact_aum 
WHERE date = (SELECT MAX(date) FROM fact_aum)
ORDER BY aum_crore DESC 
LIMIT 5;
""")

QUERY: Top 5 Fund Houses by Total AUM


,fund_house,date,aum_crore
0,SBI Mutual Fund,2025-12-31,1250000.0
1,ICICI Prudential MF,2025-12-31,1074000.0
2,HDFC Mutual Fund,2025-12-31,930000.0
3,Nippon India MF,2025-12-31,700000.0
4,Kotak Mahindra MF,2025-12-31,580000.0


In [8]:
# Query 2: Monthly Average NAV for HDFC Top 100 Direct (AMFI Code: 125497) in 2024
run_query("HDFC Top 100 Monthly Average NAV (2024)", """
SELECT strftime('%m', date) as month_num, AVG(nav) as average_nav
FROM fact_nav
WHERE amfi_code = '125497' AND date LIKE '2024%'
GROUP BY month_num
ORDER BY month_num;
""")

QUERY: HDFC Top 100 Monthly Average NAV (2024)


,month_num,average_nav
0,01,814.390332
1,02,828.244986
2,03,812.151058
3,04,818.492397
4,05,815.344113
5,06,798.837823
6,07,821.559913
7,08,792.463113
8,09,775.330853
9,10,763.115026


In [9]:
# Query 3: Industry-Wide SIP Inflows and YoY Growth
run_query("SIP Inflows and YoY Growth", """
SELECT month, sip_inflow_crore, yoy_growth_pct
FROM fact_sip_industry
ORDER BY month DESC
LIMIT 6;
""")

QUERY: SIP Inflows and YoY Growth


,month,sip_inflow_crore,yoy_growth_pct
0,2025-12,31002.0,17.17
1,2025-11,30200.0,19.27
2,2025-10,29529.0,16.61
3,2025-09,29361.0,19.80
4,2025-08,28265.0,20.04
5,2025-07,28464.0,22.00


In [10]:
# Query 4: Total Transaction Count and Amount by State
run_query("Investor Transaction Volume by State", """
SELECT state, COUNT(*) as transaction_count, SUM(amount)/10000000.0 as total_amount_crore
FROM fact_transactions
GROUP BY state
ORDER BY total_amount_crore DESC;
""")

QUERY: Investor Transaction Volume by State


,state,transaction_count,total_amount_crore
0,Punjab,2965,31.578046
1,Tamil Nadu,2806,31.517724
2,Madhya Pradesh,2931,30.831249
3,Rajasthan,2577,29.864582
4,Gujarat,2780,29.835894
5,West Bengal,2748,29.718251
6,Telangana,2718,29.021928
7,Delhi,2677,28.963340
8,Uttar Pradesh,2695,28.536887
9,Haryana,2736,27.963435


In [11]:
# Query 5: Funds with an Expense Ratio Less than 1%
run_query("Low Expense Ratio Funds (< 1.0%)", """
SELECT amfi_code, fund_house, scheme_name, expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC
LIMIT 5;
""")

QUERY: Low Expense Ratio Funds (< 1.0%)


,amfi_code,fund_house,scheme_name,expense_ratio_pct
0,118636,Nippon India MF,Nippon India Gilt Securities Fund - Regular - ...,0.55
1,100025,HDFC Mutual Fund,HDFC Short Term Debt Fund - Regular - Growth,0.56
2,120844,Kotak Mahindra MF,Kotak Liquid Fund - Regular - Growth,0.60
3,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,0.66
4,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,0.72


In [12]:
# Query 6: Top 5 Sectors with Highest Portfolio Weight Across All Funds
run_query("Top Portfolio Sectors by Average Weight", """
SELECT sector, ROUND(AVG(weight_pct), 2) as avg_weight_pct
FROM fact_portfolio
GROUP BY sector
ORDER BY avg_weight_pct DESC
LIMIT 5;
""")

QUERY: Top Portfolio Sectors by Average Weight


,sector,avg_weight_pct
0,Consumer Goods,14.18
1,Diversified,12.09
2,IT,11.39
3,Utilities,11.06
4,FMCG,10.91


In [13]:
# Query 7: KYC Status Distribution and Transaction Volumes
run_query("Investor KYC Status Analysis", """
SELECT kyc_status, COUNT(DISTINCT investor_id) as investor_count, SUM(amount)/10000000.0 as total_amount_crore
FROM fact_transactions
GROUP BY kyc_status;
""")

QUERY: Investor KYC Status Analysis


,kyc_status,investor_count,total_amount_crore
0,Pending,2013,28.740805
1,Verified,4961,323.417238


In [14]:
# Query 8: Monthly Transaction Type Distribution
run_query("Transaction Distribution by Type (Last 6 Months)", """
SELECT strftime('%Y-%m', date) as month, type, COUNT(*) as tx_count, SUM(amount)/10000000.0 as amount_crore
FROM fact_transactions
GROUP BY month, type
ORDER BY month DESC, type ASC
LIMIT 12;
""")

QUERY: Transaction Distribution by Type (Last 6 Months)


,month,type,tx_count,amount_crore
0,2025-05,Lumpsum,474,12.007315
1,2025-05,Redemption,287,6.960821
2,2025-05,SIP,1201,1.397301
3,2025-04,Lumpsum,473,11.957111
4,2025-04,Redemption,300,7.635476
5,2025-04,SIP,1086,1.213200
6,2025-03,Lumpsum,466,12.803567
7,2025-03,Redemption,311,7.818237
8,2025-03,SIP,1196,1.344936
9,2025-02,Lumpsum,445,10.673754


In [15]:
# Query 9: Best 3-Year CAGR Funds with Risk Profiles
run_query("Top 5 Funds by 3-Year Return & Risk Category", """
SELECT f.amfi_code, f.scheme_name, f.risk_category, p.return_3yr
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
ORDER BY p.return_3yr DESC
LIMIT 5;
""")

QUERY: Top 5 Funds by 3-Year Return & Risk Category


,amfi_code,scheme_name,risk_category,return_3yr
0,119598,SBI Small Cap Fund - Regular Plan - Growth,Very High,23.39
1,119599,SBI Small Cap Fund - Direct Plan - Growth,Very High,23.14
2,101207,ABSL Small Cap Fund - Regular - Growth,Very High,22.38
3,119095,Axis Small Cap Fund - Regular - Growth,Very High,20.98
4,118634,Nippon India Small Cap Fund - Regular - Growth,Very High,20.15


In [16]:
# Query 10: 30-Day Moving Average NAV of SBI Bluechip (AMFI Code: 119551) in Jan-Mar 2024
run_query("30-Day Moving Average NAV for SBI Bluechip", """
SELECT date, nav, 
       ROUND(AVG(nav) OVER (ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW), 2) as moving_avg_30d
FROM fact_nav
WHERE amfi_code = '119551' AND date BETWEEN '2024-01-01' AND '2024-03-31'
ORDER BY date
LIMIT 10;
""")

QUERY: 30-Day Moving Average NAV for SBI Bluechip


,date,nav,moving_avg_30d
0,2024-01-01,70.8078,70.81
1,2024-01-02,71.1846,71.00
2,2024-01-03,70.8063,70.93
3,2024-01-04,71.2003,71.00
4,2024-01-05,70.7591,70.95
5,2024-01-06,70.7591,70.92
6,2024-01-07,70.7591,70.90
7,2024-01-08,69.6776,70.74
8,2024-01-09,68.7201,70.52
9,2024-01-10,68.7847,70.35


In [17]:
# Close database connection
conn.close()
print("Database connection closed.")

Database connection closed.
